# Validation Metrics

This notebook presents validation metrics for a biomedical knowledge graph built from 1,000 PubMed abstracts, covering named entity recognition (NER), entity linking to UMLS and relationship extraction.

**Purpose:** Assess the quality of entity extraction, linking to medical terminology (UMLS), and relationship extraction before deploying to Neo4j or using for research.

**Validation Corpus:**
- **Size:** 1,000 abstracts
- **Search term:** "breast cancer AND targeted therapy"
- **Build date:** 2025-11-18 14:26:11
- **Pipeline:** PubMed $\rightarrow$ NER (3 scispaCy models) $\rightarrow$ UMLS linking $\rightarrow$ Co-occurrence extraction

**Metrics Reported:**
1. **Entity Extraction:** Total entities and how many linked to standard medical terminology (UMLS)
2. **Linking Coverage:** Percentage of entity mentions successfully standardized
3. **Entity Normalization Ratio:** How many text variants map to each UMLS concept (measures synonym consolidation)
4. **Entity Types:** Distribution across biomedical categories (genes, chemicals, diseases, etc.)
5. **Non-Standard Entities:** Analysis of entities that didn't link to UMLS
6. **Relationships:** Co-occurrence statistics and evidence strength

**For methodology and context, see:** `docs/validation.md`


## Setup

### Build Validation Corpus

The validation corpus is built with:

```bash
biomed-agent run-pipeline \
  --search-term "breast cancer AND targeted therapy" \
  --size 1000 \
  --output-dir data/validation
```

This creates:
- `data/validation/corpus.db` - PubMed abstracts
- `data/validation/nlp.db` - NER + UMLS linking results  
- `data/validation/kg.db` - Knowledge graph with co-occurrence relationships

In [18]:
import sqlite3
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

# Configure database paths
# Point these to the kg.db and nlp.db from your pipeline run
# Default: validation corpus in data/validation/ (relative to project root)
KG_DB_PATH = "../data/validation/kg.db"
NLP_DB_PATH = "../data/validation/nlp.db"

# Verify files exist
kg_path = Path(KG_DB_PATH)
nlp_path = Path(NLP_DB_PATH)

if not kg_path.exists():
    raise FileNotFoundError(f"KG database not found: {kg_path}\nSee setup instructions above.")
if not nlp_path.exists():
    raise FileNotFoundError(f"NLP database not found: {nlp_path}\nSee setup instructions above.")

print(f"Using KG database: {kg_path}")
print(f"Using NLP database: {nlp_path}")

# Open database connections (closed at end of notebook)
conn_kg = sqlite3.connect(KG_DB_PATH)
conn_nlp = sqlite3.connect(NLP_DB_PATH)
cursor_kg = conn_kg.cursor()
cursor_nlp = conn_nlp.cursor()


Using KG database: ../data/validation/kg.db
Using NLP database: ../data/validation/nlp.db


## Entity Statistics

**Metrics:**
- Total unique entities extracted from the corpus
- **UMLS entities**: Linked to standard medical terminology (e.g., "p53" $\rightarrow$ UMLS concept C0080055)
- **Custom entities**: Not linked to UMLS (may be novel terms, abbreviations, or NER errors)
- Distribution across biomedical entity types (genes, chemicals, diseases, etc.)

In [19]:
# Total entities
cursor_kg.execute("SELECT COUNT(*) FROM entity")
total_entities = cursor_kg.fetchone()[0]

# UMLS entities (have umls_cui)
cursor_kg.execute("SELECT COUNT(*) FROM entity WHERE umls_cui IS NOT NULL")
umls_entities = cursor_kg.fetchone()[0]

# Custom entities (no umls_cui)
custom_entities = total_entities - umls_entities

# Entity type distribution
cursor_kg.execute("""
    SELECT entity_type, COUNT(*) as count
    FROM entity
    GROUP BY entity_type
    ORDER BY count DESC
""")
entity_types = cursor_kg.fetchall()

print("=" * 60)
print("ENTITY STATISTICS")
print("=" * 60)
print(f"Total entities: {total_entities:,}")
print(f"UMLS entities: {umls_entities:,} ({umls_entities/total_entities*100:.1f}%)")
print(f"Custom entities: {custom_entities:,} ({custom_entities/total_entities*100:.1f}%)")
print()
print("Entity Type Distribution:")
print("-" * 60)
for entity_type, count in entity_types:
    pct = count / total_entities * 100
    print(f"{entity_type:25} {count:8,} ({pct:5.1f}%)")


ENTITY STATISTICS
Total entities: 5,104
UMLS entities: 3,339 (65.4%)
Custom entities: 1,765 (34.6%)

Entity Type Distribution:
------------------------------------------------------------
chemical                     1,814 ( 35.5%)
gene                         1,126 ( 22.1%)
disease                        907 ( 17.8%)
cell_type                      293 (  5.7%)
anatomy                        241 (  4.7%)
organism                       166 (  3.3%)
pathology                      116 (  2.3%)
biological_process             111 (  2.2%)
sequence_feature               103 (  2.0%)
substance                       94 (  1.8%)
cellular_component              77 (  1.5%)
procedure                       52 (  1.0%)
amino_acid                       4 (  0.1%)


**Observations:**
- Entity type distribution reflects the corpus focus (breast cancer + targeted therapy): chemicals (35.5%), genes (22.1%), and diseases (17.8%) dominate
- The top 3 categories account for 75.4% of all entities
- About 2/3 of entities (65.4%) link to UMLS; the remaining 1/3 are custom entities (domain-specific abbreviations, novel terms, or extraction artifacts)


## UMLS Linking 

UMLS linking merges textual variations of an abstract entity to a canonical UMLS concept, *e.g.*, "breast cancer" and "malignant neoplasm of breast" both map to "C0006142". 

This entity normalization improves robust knowledge extraction and enables linking to other data sources.

### Coverage

**Metrics:** 

- **Mention-level**: Out of all entity mentions in the text, what % got linked?
  - Weighted by frequency: common terms like "cancer" or "p53" count more
  - Example: If "cancer" appears 100 times and links, that's 100 linked mentions
  
- **Text-level**: Out of all unique text strings, what % got linked?
  - Treats all terms equally regardless of frequency
  - Example: "cancer" (100 mentions) and "cancre" (1 typo) each count as one unique text

- **Entity Normalization Ratio**: On average, how many linked unique text strings refer to the same UMLS concept?
  - Higher value implies stronger semantic compression
  - Example: If 10 text variants ("breast cancer", "BC", "breast carcinoma"...) map to 5 UMLS concepts, the ratio is 2.0x

**Note:** The multi-model NER ensemble extracts overlapping entity spans from the same text (e.g., "breast cancer" at positions 27-40 and "breast cancer patients" at 27-49 in the same sentence). Both are counted as mentions, though they consolidate to a single entity when they link to the same UMLS concept. 


In [20]:
# Mention-level coverage
cursor_nlp.execute("SELECT COUNT(*) FROM extracted_entities")
total_mentions = cursor_nlp.fetchone()[0]

cursor_nlp.execute("SELECT COUNT(*) FROM extracted_entities WHERE umls_cui IS NOT NULL")
linked_mentions = cursor_nlp.fetchone()[0]

# Text-level coverage (unique texts)
cursor_nlp.execute("SELECT COUNT(DISTINCT text) FROM extracted_entities")
total_unique_texts = cursor_nlp.fetchone()[0]

cursor_nlp.execute("""
    SELECT COUNT(DISTINCT text)
    FROM extracted_entities
    WHERE umls_cui IS NOT NULL
""")
linked_texts = cursor_nlp.fetchone()[0]

# UMLS CUI count (unique concepts)
cursor_nlp.execute("""
    SELECT COUNT(DISTINCT umls_cui)
    FROM extracted_entities
    WHERE umls_cui IS NOT NULL
""")
unique_cuis = cursor_nlp.fetchone()[0]

# Entity normalization: how many text variants map to each UMLS concept
variants_per_concept = linked_texts / unique_cuis if unique_cuis > 0 else 0

print("=" * 60)
print("UMLS LINKING COVERAGE")
print("=" * 60)
print()
print("Mention-Level Coverage (Primary Metric):")
print(f"  Total mentions: {total_mentions:,}")
print(f"  Linked mentions: {linked_mentions:,} ({linked_mentions/total_mentions*100:.1f}%)")
print(f"  Custom mentions: {total_mentions - linked_mentions:,} ({(total_mentions-linked_mentions)/total_mentions*100:.1f}%)")
print()
print("Text-Level Coverage:")
print(f"  Unique texts: {total_unique_texts:,}")
print(f"  Linked texts: {linked_texts:,} ({linked_texts/total_unique_texts*100:.1f}%)")
print()
print("Entity Normalization:")
print(f"  Unique UMLS concepts: {unique_cuis:,}")
print(f"  Text variants per concept: {variants_per_concept:.2f}x")
print(f"  (Average {variants_per_concept:.1f} different text strings map to each concept)")


UMLS LINKING COVERAGE

Mention-Level Coverage (Primary Metric):
  Total mentions: 28,806
  Linked mentions: 23,873 (82.9%)
  Custom mentions: 4,933 (17.1%)

Text-Level Coverage:
  Unique texts: 7,652
  Linked texts: 5,833 (76.2%)

Entity Normalization:
  Unique UMLS concepts: 3,399
  Text variants per concept: 1.72x
  (Average 1.7 different text strings map to each concept)


**Interpretation:**
- **High mention-level coverage (82.9%)** - Majority of extracted mentions successfully standardize to UMLS terminology, indicating strong knowledge graph quality.
- **Lower text-level coverage (76.2%)** - Expected for specialized corpora due to domain-specific abbreviations and rare terms (see Custom Entity Analysis below).
- **Low normalization ratio (1.72x)** - Reflects limited synonym observations: 70.7% of UMLS concepts have only 1 text variant (see variant distribution below). Small corpus + specialized terminology = few synonyms observed. Higher ratio expected for increased corpus size.

### Distribution of text variants per UMLS concept

**Metric:** Number of distinct text strings mapping to each UMLS concept (indicates synonym consolidation). Higher distributions indicate stronger UMLS linking consolidation.

In [21]:
# Distribution of text variants per UMLS concept

sql = """
SELECT 
    umls_cui,
    COUNT(DISTINCT text) as variant_count
FROM extracted_entities
WHERE umls_cui IS NOT NULL
GROUP BY umls_cui
ORDER BY variant_count DESC
"""
df_variants = pd.read_sql_query(sql, conn_nlp)

# Summary statistics
print("=" * 70)
print("TEXT VARIANT DISTRIBUTION PER UMLS CONCEPT")
print("=" * 70)
print(f"Total UMLS concepts: {len(df_variants):,}")
print(f"Average variants per concept: {df_variants['variant_count'].mean():.2f}")
print(f"Median variants per concept: {df_variants['variant_count'].median():.0f}")
print(f"Max variants: {df_variants['variant_count'].max()}")
print()

# Show distribution
variant_dist = df_variants['variant_count'].value_counts().sort_index()
print(f"{'Variants':>10} {'# CUIs':>10} {'Percentage':>12}")
print("-" * 70)

for variants, count in variant_dist.head(10).items():
    pct = count / len(df_variants) * 100
    print(f"{int(variants):>10} {count:>10,} {pct:>11.1f}%")

if len(variant_dist) > 10:
    print(f"{'...':<10} {'...':<10} {'...':<12}")

print()
print("Top 5 concepts with most variants:")
print("-" * 70)
top_5 = df_variants.nlargest(5, 'variant_count')
for _, row in top_5.iterrows():
    cursor_nlp.execute(
        "SELECT umls_preferred_name FROM extracted_entities WHERE umls_cui = ? LIMIT 1",
        (row['umls_cui'],)
    )
    name = cursor_nlp.fetchone()[0]
    print(f"  {row['umls_cui']} - {name}: {int(row['variant_count'])} variants")

TEXT VARIANT DISTRIBUTION PER UMLS CONCEPT
Total UMLS concepts: 3,399
Average variants per concept: 1.72
Median variants per concept: 1
Max variants: 77

  Variants     # CUIs   Percentage
----------------------------------------------------------------------
         1      2,404        70.7%
         2        570        16.8%
         3        194         5.7%
         4         86         2.5%
         5         42         1.2%
         6         31         0.9%
         7         18         0.5%
         8         10         0.3%
         9         11         0.3%
        10          8         0.2%
...        ...        ...         

Top 5 concepts with most variants:
----------------------------------------------------------------------
  C0006142 - Malignant neoplasm of breast: 77 variants
  C0006826 - Malignant Neoplasms: 43 variants
  C0086418 - Homo sapiens: 42 variants
  C0006141 - Breast: 41 variants
  C0027651 - Neoplasms: 31 variants


**Observations:**
- Most concepts (70.7%) have only 1 text variant - limited synonym observation in this corpus
- Average of 1.72 text variants per concept, median of 1
- A few concepts accumulate many variants (max: 77), creating outliers in the distribution
- The low consolidation ratio may reflect the small corpus size (1,000 abstracts) and specialized terminology with few observed synonyms

### Text Variant Composition

The following query shows text variants from the NLP database (`extracted_entities`), which preserves raw extraction output before consolidation. Case variants like "Breast cancer..." vs "breast cancer..." are normalized during knowledge graph construction (entities consolidate by UMLS CUI).

In [22]:
# Case Study: C0006142 (Malignant neoplasm of breast) - Sample Variants

sql = """
SELECT DISTINCT text, entity_type 
FROM extracted_entities 
WHERE umls_cui = 'C0006142'
ORDER BY LENGTH(text) DESC
"""
variants = pd.read_sql_query(sql, conn_nlp)

print("=" * 60)
print("C0006142 (Malignant neoplasm of breast) - Sample Variants")
print("=" * 60)
print()
print(f"Number of variants: {len(variants)}\n")

print("Text Variants")
print("=" * 70)
for _, row in variants.iterrows():
    print(f"  - '{row.text}' ({row.entity_type})")

C0006142 (Malignant neoplasm of breast) - Sample Variants

Number of variants: 77

Text Variants
  - 'breast cancer type 1 susceptibility protein' (disease)
  - 'breast cancer stem cell phenotypes.' (disease)
  - 'EO771 orthotopic breast cancer' (disease)
  - 'breast cancer liver metastasis' (disease)
  - 'breast cancer liver metastases' (disease)
  - 'osteolytic breast cancer bone' (disease)
  - 'triple‑negative breast cancer' (disease)
  - 'breast cancer patient samples' (disease)
  - 'Breast cancer cell-derived AM' (disease)
  - 'breast cancer cell-derived AM' (disease)
  - 'breast cancer cell line MCF-7' (disease)
  - 'breast cancer lung metastasis' (disease)
  - 'breast cancer patient tumors' (disease)
  - 'breast cancer cell membranes' (disease)
  - 'ER⁺ breast cancer cell lines' (disease)
  - 'breast cancer tissue samples' (disease)
  - 'Breast cancer tissue samples' (disease)
  - 'Breast cancer tumor tissues' (disease)
  - 'breast cancer tumorigenesis' (disease)
  - 'left breas

The UMLS linker consolidates compositional phrases to base concepts. Above: 77 text variants containing "breast cancer" all map to C0006142, including phrases that may be semantically distinct (cell lines, animal models, metastases).

**Observations:**
- This grouping strategy increases recall: a query for "breast cancer" retrieves all related research contexts
- Trade-off: reduced granularity, as distinct sub-concepts are merged into the same CUI

## Custom Entity Analysis

**Custom entities** are terms extracted by NER but not linked to UMLS standard terminology.

**Reasons for non-linking:**
- Novel scientific terminology not yet in UMLS
- Domain-specific abbreviations (e.g., "TNBC" = triple-negative breast cancer)
- Non-standard phrasings
- NER false positives (extraction errors)

**Analysis:** Top 20 most frequent custom entities by mention count.


In [23]:
# Get top custom entities by mention frequency
cursor_nlp.execute("""
    SELECT text, entity_type, COUNT(*) as mention_count
    FROM extracted_entities
    WHERE umls_cui IS NULL
    GROUP BY text, entity_type
    ORDER BY mention_count DESC
    LIMIT 20
""")
top_custom = cursor_nlp.fetchall()

print("=" * 80)
print("TOP 20 CUSTOM ENTITIES (by mention frequency)")
print("=" * 80)
print(f"{'Text':<40} {'Type':<20} {'Mentions':>10}")
print("-" * 80)
for text, entity_type, count in top_custom:
    # Truncate long texts
    display_text = text[:37] + "..." if len(text) > 40 else text
    print(f"{display_text:<40} {entity_type:<20} {count:>10,}")

TOP 20 CUSTOM ENTITIES (by mention frequency)
Text                                     Type                   Mentions
--------------------------------------------------------------------------------
TNBC                                     disease                     653
cells                                    cell_type                   568
death                                    disease                     130
Cancer                                   disease                      91
TNBC cells                               cell_type                    53
ADCs                                     disease                      51
T-DXd                                    disease                      50
MBC                                      chemical                     36
TNBC patients                            disease                      34
±                                        chemical                     28
HER2-low                                 cell_type                    

**Observations:**
- High-frequency custom entities could be candidates for manual review
- Some unlinked terms may be novel/specialized biomedical concepts (e.g., TNBC, ADCs, T-DXd)
- Very short or generic terms might be NER false positives (e.g., "±", "integrated")
- Note: Generic terms like "cells", "death", "Cancer" appear here but are filtered during Neo4j migration via stopword filtering (see `src/biomed_kg_agent/kg/filtering.py`)

## Relationship Statistics

**Analysis**: Entity co-occurrence relationships in the knowledge graph.

**Co-occurrence = entities mentioned together in the same sentence**
- Example: "p53 mutations are common in breast cancer" creates a co-occurrence between p53 (gene) and breast cancer (disease)

**Evidence metrics:**
- **docs_count**: How many different papers mention both entities together (higher = more reliable)
- **sent_count**: Total sentences where they co-occur (includes multiple mentions in same paper)


In [24]:
# Total relationships
cursor_kg.execute("SELECT COUNT(*) FROM cooccurrence")
total_rels = cursor_kg.fetchone()[0]

# Evidence strength distribution (by docs_count)
# Higher docs_count = relationship observed in more independent papers
cursor_kg.execute("""
    SELECT 
        CASE 
            WHEN docs_count >= 10 THEN 'Very Strong (10+ papers)'
            WHEN docs_count >= 5 THEN 'Strong (5-9 papers)'
            WHEN docs_count >= 3 THEN 'Moderate (3-4 papers)'
            WHEN docs_count >= 2 THEN 'Weak (2 papers)'
            ELSE 'Single-source (1 paper)'
        END as evidence_tier,
        COUNT(*) as count
    FROM cooccurrence
    GROUP BY evidence_tier
    ORDER BY MIN(docs_count) DESC
""")
evidence_dist = cursor_kg.fetchall()

# Average metrics
cursor_kg.execute("""
    SELECT 
        AVG(docs_count) as avg_docs,
        AVG(sent_count) as avg_sents,
        MAX(docs_count) as max_docs,
        MAX(sent_count) as max_sents
    FROM cooccurrence
""")
avg_docs, avg_sents, max_docs, max_sents = cursor_kg.fetchone()

print("=" * 60)
print("RELATIONSHIP STATISTICS")
print("=" * 60)
print(f"Total co-occurrence relationships: {total_rels:,}")
print(f"Average documents per relationship: {avg_docs:.1f}")
print(f"Average sentences per relationship: {avg_sents:.1f}")
print(f"Max documents for any relationship: {int(max_docs):,}")
print(f"Max sentences for any relationship: {int(max_sents):,}")
print()
print("Evidence Strength Distribution:")
print("(Based on number of documents independently observing the co-occurrence)")
print("-" * 60)
for tier, count in evidence_dist:
    pct = count / total_rels * 100
    print(f"{tier:30} {count:8,} ({pct:5.1f}%)")



RELATIONSHIP STATISTICS
Total co-occurrence relationships: 31,955
Average documents per relationship: 1.4
Average sentences per relationship: 1.5
Max documents for any relationship: 142
Max sentences for any relationship: 153

Evidence Strength Distribution:
(Based on number of documents independently observing the co-occurrence)
------------------------------------------------------------
Very Strong (10+ papers)            254 (  0.8%)
Strong (5-9 papers)                 432 (  1.4%)
Moderate (3-4 papers)               865 (  2.7%)
Weak (2 papers)                   1,785 (  5.6%)
Single-source (1 paper)          28,619 ( 89.6%)


**Observations:**
- Most relationships (89.6%) are observed in only a single paper - may reflect the limited corpus size (1,000 abstracts)
- Multi-paper relationships (10.4%, or 3,336 relationships) appear across multiple independent sources
- Average of 1.4 documents per relationship suggests limited overlap in this corpus
- Strong-evidence relationships (10+ papers) represent 0.8% (254 relationships) - these may be well-established associations
- Single-paper relationships could represent either novel findings or spurious co-occurrences (or NER artifacts)

In [25]:
# Close connections
conn_kg.close()
conn_nlp.close()

## Discussion

### Filtering Considerations

The knowledge graph contains relationships with varying evidence strength (89.6% single-paper, 10.4% multi-paper). The choice between including all relationships versus filtering by document count depends on research objectives:
- Filtering for multi-paper relationships reduces noise but may exclude novel findings
- Including all relationships preserves completeness but introduces potential spurious co-occurrences

### Pipeline Characteristics

This validation assesses a co-occurrence-based knowledge graph with the following design characteristics:

**Design characteristics and trade-offs:**
1. **Co-occurrence extraction**: Relationships based on sentence-level co-occurrence, not semantic parsing. "Drug X treats Disease Y" and "Drug X failed to treat Disease Y" create the same relationship.
2. **Negation handling**: Negative assertions ("not associated with", "no effect on") treated as co-occurrences.
3. **Relationship directionality**: Co-occurrence is symmetric; causal/hierarchical relationships not distinguished.
4. **Abbreviation handling**: Domain-specific abbreviations (TNBC, ADCs) may not link to UMLS without manual curation or external dictionaries.
5. **Compositional phrase linking**: UMLS linker maps compositional phrases to base concepts, increasing recall at the cost of granularity.

### Summary

This validation provides baseline quality metrics for a 1,000 abstract biomedical knowledge graph focused on breast cancer and targeted therapy:
- UMLS standardization: 82.9% mention-level coverage, 76.2% text-level coverage
- Relationship evidence: 89.6% single-paper relationships, 10.4% multi-paper relationships
- Custom entities: 34.6% unlinked, including domain-specific abbreviations and potentially novel terms

The interpretation of these metrics depends on research objectives - whether prioritizing established, well-replicated knowledge or exploring emerging, specialized findings.